# Decay plot and b-value maps

Two figures from the calibration scans (no leave-one-out is needed as both show measured data):

1. **Signal decay** - ln(S/S0) against b-value at two marked voxels, for each diffusion direction, on top of the
   anatomy and the decay expected for water.
2. **b-value maps** - the measured correction factor (effective b / nominal b), averaged over the repeats of each
   sequence and smoothed with a B-spline, for each b-value and direction, with the values at five voxels plotted
   against b-value.

Scans, masks and settings come from `config.py` (`SCAN_TYPE`); the per-scan b-value maps are read from the cache
written by `estimate_calibration.py`.

In [ ]:
# Project settings and paths: config.py is in the project root (this also makes the modules in src/ importable)
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())))
import config

# General libraries
import json
import numpy as np
import pandas as pd
import nibabel as nib
import ants

# Plotting libraries
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import colorcet

# Custom functions
import registration as reg
import temperature as temp
import calib_utils as cal

In [ ]:
REF_TEMP = 22 # C - temperature the signal decay is scaled to and the water reference line is drawn for

config.PLOT_DIR.mkdir(parents=True, exist_ok=True)

### LOAD DATA

In [ ]:
# Unpack data from files
scans_nii = nib.load(config.SCANS_FNAME)
scans_data = scans_nii.get_fdata().astype(np.float32)
scan_aff = scans_nii.affine
scan_spacing = reg.spacing_from_affine(scan_aff)

df = pd.read_hdf(config.PARAM_FNAME, key="df", mode="r")
df["row"] = np.arange(df.shape[0]) # position in the parameter file, scan files and cached maps
scans_reshuffled = np.transpose(scans_data, [3,0,1,2])
df["image"] = list(scans_reshuffled)
df["adc"] = df.apply(lambda row: temp.water_diffusivity(temp=row["temperature"], polynomial=True), axis=1)

preprocess_mask = nib.load(config.MASK_FNAME).get_fdata().astype(bool)
calib_mask = cal.shrink_mask_mm(preprocess_mask, scan_spacing, margin_mm=config.SHRINK_DIST_MM)

# Per-scan b-value maps from estimate_calibration.py. Volume i is row i of the parameter file.
bmap_nii = nib.load(config.BMAP_FNAME)
bmap_meta = json.loads(next(e for e in bmap_nii.header.extensions if e.get_code() == 40).content.decode("utf-8"))
if bmap_meta["scan_type"] != config.SCAN_TYPE:
    raise ValueError(f"Cached b-value maps are from the {bmap_meta['scan_type']} scans, config.SCAN_TYPE is {config.SCAN_TYPE}")
bmap_4d = bmap_nii.get_fdata()

In [ ]:
df.sort_values("scan_time", inplace=True, ignore_index=True)

# Put the b=0 image on b=0 rows, NaN elsewhere
b0_threshold = config.B0_THRESHOLD
s0_avg_img = np.mean(df[df["b_value"] < b0_threshold]["image"].values, axis=0).astype(np.float32)

# Forward-fill so each DWI scan gets the most recent preceding b=0 image
df["s0_image"] = df["image"].where(df["b_value"] < b0_threshold)
df["s0_image"] = df["s0_image"].ffill()

# Signal decay relative to the b=0 image, scaled to the reference temperature
df["minus_bd"] = df.apply(lambda row: cal.minus_bd_map(row["image"], row["s0_image"], mask=preprocess_mask), axis=1)
df["ref_minus_bd"] = df["minus_bd"] * temp.water_diffusivity(REF_TEMP) / df["adc"]

# Measured b-value map of each scan, from the cache
df["bmap"] = df["row"].apply(lambda i: bmap_4d[..., i])

# Keep only diffusion-weighted scans
dwi_df = df[df["b_value"] > b0_threshold].copy().reset_index(drop=True)
dwi_df["cmap_meas"] = dwi_df["bmap"]/dwi_df["b_value"]

### SIGNAL DECAY

In [ ]:
def summarise_signal_by_bvalue(bvals, signal):
    bvals = np.asarray(bvals)
    signal = np.asarray(signal)

    unique_bvals = np.sort(np.unique(bvals))

    mean_vals = []
    q05_vals = []
    q95_vals = []

    for b in unique_bvals:
        vals = signal[bvals == b]
        mean_vals.append(np.mean(vals))
        q05_vals.append(np.percentile(vals, 5))
        q95_vals.append(np.percentile(vals, 95))

    return {
        "bvals": unique_bvals,
        "mean": np.array(mean_vals),
        "q05": np.array(q05_vals),
        "q95": np.array(q95_vals),
    }


# ------------------
# voxel definitions from script 2
# ------------------
img_centre = np.array(preprocess_mask.shape) // 2
dz = 0
i_slice = img_centre[2] + dz

i_v1 = img_centre + np.array([-21,  1, dz])
i_v2 = img_centre + np.array([ 14, 18, dz])

voxel_indices = [i_v1, i_v2]
voxel_colors = ["b", "r"]
voxel_labels = ["P1", "P2"]


def orig_to_plot_coords_rc(row, col, shape_2d):
    """
    Map original 2D array coordinates (row, col)
    to displayed plot coordinates (x, y) for

        np.rot90(arr, k=1)[:, :-1]

    using imshow default origin='upper'.
    """
    n_rows, n_cols = shape_2d
    x = row
    y = n_cols - 1 - col
    return x, y

fs_axtitle = 12
fs_plottitle = 20
fs_ticks = 12
fs_legend = 12

fig, ax = plt.subplots(
    1, 4,
    figsize=(18, 4.5),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [0.7, 1, 1, 1]},
)
fig.set_constrained_layout_pads(wspace=0.1, hspace=0.05)

# ------------------
# anatomy panel
# ------------------
s0_avg_img_a = ants.from_numpy(s0_avg_img, spacing=scan_spacing)
s0_avg_img_bc = ants.n4_bias_field_correction(s0_avg_img_a, 
                                              mask=ants.get_mask(s0_avg_img_a), 
                                              rescale_intensities=True, 
                                              shrink_factor=2,
                                              convergence={"iters": [100, 50, 50], "tol": 1e-7},
                                              spline_param=175,
                                              ).numpy()
slice_raw = s0_avg_img_bc[:, :, i_slice]
plot_slice = np.rot90(slice_raw, k=1)[:, :-1]

vmin = np.percentile(plot_slice, 1)
vmax = np.percentile(plot_slice, 99.9)

ax[0].imshow(plot_slice, vmin=vmin, vmax=vmax, cmap="gray", interpolation=None)
ax[0].set_axis_off()

# overlay marker positions using plot markers
for idx, color in zip(voxel_indices, voxel_colors):
    x, y = orig_to_plot_coords_rc(idx[0], idx[1], slice_raw.shape)
    ax[0].plot(
        x,
        y,
        marker="o",
        ms=4.5,
        color=color,
        mec="w",
        mew=0.5,
        linestyle="none",
    )


dur = 0.035
plot_df = df[df["dwi_grad_duration"]==dur].copy()

for i, a in enumerate(plot_df["dwi_direction"].unique(), start=1):

    sub_df = plot_df[plot_df["dwi_direction"] == a].copy()
    bvals = sub_df["b_value"].values
    ref_mbd = np.stack(sub_df["ref_minus_bd"].values, axis=-1)   # X,Y,Z,N

    for idx, color, label in zip(voxel_indices, voxel_colors, voxel_labels):
        voxel_signal = ref_mbd[idx[0], idx[1], idx[2], :]
        voxel_data = summarise_signal_by_bvalue(bvals, voxel_signal)

        ax[i].plot(
            voxel_data["bvals"],
            voxel_data["mean"],
            color=color,
            marker="o",
            markersize=4.5,
            mec="w",
            mew=0.5,
            linewidth=1.2,
            label=label,
        )
        ax[i].fill_between(
            voxel_data["bvals"],
            voxel_data["q05"],
            voxel_data["q95"],
            color=color,
            alpha=0.2,
            linewidth=0,
        )

    # Reference water diffusion line
    bvals_ref = np.sort(np.unique(bvals))
    ax[i].plot(
        bvals_ref,
        -bvals_ref * temp.water_diffusivity(REF_TEMP),
        dashes=[3, 2],
        c="grey",
        label=r"$ADC_{WATER}\ (T=20 \degree C)$"
    )

    ax[i].set_title(rf"${a}$", fontsize=fs_plottitle, pad=12)
    ax[i].set_xlabel(r"$b\ (s/mm^2)$", fontsize=fs_ticks)
    ax[i].set_xlim(-50, 850)
    ax[i].set_ylim(-3, 0.5)
    ax[i].grid(alpha=0.25)
    ax[i].tick_params(axis='both', labelsize=fs_ticks)
    ax[i].spines.right.set_visible(False)
    ax[i].spines.top.set_visible(False)

    if i != 1:
        ax[i].set_yticklabels([])

    if i == 1:
        ax[i].legend(loc="lower left", fontsize=fs_legend)

ax[1].set_ylabel(r"$ln(S/S_0)$", fontsize=fs_axtitle)

plot_fname = config.PLOT_DIR / "calibration_b_value_decay_plots.png"
plt.savefig(plot_fname, dpi=600, format="png", bbox_inches="tight", pad_inches=0.1)

In [ ]:
# ----------------------------------------------------------------------------------------------------
# Version of the decay figure with the b=800 scans: the S0 image (left), the b=800 scan of each axis with the bias
# field estimated from the S0 image applied (top row) and the signal decay at the marked voxels (bottom row).
# ----------------------------------------------------------------------------------------------------
b_show = 800    # b-value of the scans shown

# Bias field of the S0 image (the N4 bias field correction in the cell above), applied to the diffusion-weighted images
bias_field = np.ones_like(s0_avg_img)
np.divide(s0_avg_img_bc, s0_avg_img, out=bias_field, where=s0_avg_img > 0)

# b=800 scan of each axis: mean over the repeats, bias field applied
dwi_slices = {}
for a in plot_df["dwi_direction"].unique():
    sub_df = plot_df[(plot_df["dwi_direction"] == a) & (np.abs(plot_df["b_value"] - b_show) < 50)]
    dwi_img = np.mean(np.stack(sub_df["image"].values), axis=0) * bias_field
    dwi_slices[a] = np.rot90(dwi_img[:, :, i_slice], k=1)[:, :-1]

# Common intensity scale for the three diffusion-weighted images
dwi_all = np.concatenate([s.ravel() for s in dwi_slices.values()])
dwi_vmin, dwi_vmax = np.percentile(dwi_all, 1), np.percentile(dwi_all, 99.9)

fig = plt.figure(figsize=(18, 7.5), constrained_layout=True)
gs = fig.add_gridspec(2, 4, width_ratios=[0.7, 1, 1, 1], height_ratios=[1, 1])
fig.set_constrained_layout_pads(wspace=0.1, hspace=0.05)

# ------------------
# S0 image (first column, nothing below it)
# ------------------
ax_s0 = fig.add_subplot(gs[0, 0])
s0_slice = np.rot90(s0_avg_img_bc[:, :, i_slice], k=1)[:, :-1]
ax_s0.imshow(s0_slice, vmin=np.percentile(s0_slice, 1), vmax=np.percentile(s0_slice, 99.9), cmap="gray", interpolation=None)
ax_s0.set_axis_off()
ax_s0.set_title(r"$S_0$", fontsize=fs_plottitle, pad=12)

def mark_voxels(ax):
    for idx, color in zip(voxel_indices, voxel_colors):
        x, y = orig_to_plot_coords_rc(idx[0], idx[1], s0_avg_img_bc[:, :, i_slice].shape)
        ax.plot(x, y, marker="o", ms=4.5, color=color, mec="w", mew=0.5, linestyle="none")

mark_voxels(ax_s0)

# ------------------
# b=800 images (top row) and signal decay (bottom row) for each axis
# ------------------
for i, a in enumerate(plot_df["dwi_direction"].unique(), start=1):

    ax_img = fig.add_subplot(gs[0, i])
    ax_img.imshow(dwi_slices[a], vmin=dwi_vmin, vmax=dwi_vmax, cmap="gray", interpolation=None)
    ax_img.set_axis_off()
    ax_img.set_title(rf"${a}$", fontsize=fs_plottitle, pad=12)
    mark_voxels(ax_img)

    ax_line = fig.add_subplot(gs[1, i])
    sub_df = plot_df[plot_df["dwi_direction"] == a].copy()
    bvals = sub_df["b_value"].values
    ref_mbd = np.stack(sub_df["ref_minus_bd"].values, axis=-1)   # X,Y,Z,N

    for idx, color, label in zip(voxel_indices, voxel_colors, voxel_labels):
        voxel_data = summarise_signal_by_bvalue(bvals, ref_mbd[idx[0], idx[1], idx[2], :])

        ax_line.plot(voxel_data["bvals"], voxel_data["mean"], color=color, marker="o", markersize=4.5,
                     mec="w", mew=0.5, linewidth=1.2, label=label)
        ax_line.fill_between(voxel_data["bvals"], voxel_data["q05"], voxel_data["q95"],
                             color=color, alpha=0.2, linewidth=0)

    # Reference water diffusion line
    bvals_ref = np.sort(np.unique(bvals))
    ax_line.plot(bvals_ref, -bvals_ref * temp.water_diffusivity(REF_TEMP), dashes=[3, 2], c="grey",
                 label=r"$ADC_{WATER}\ (T=20 \degree C)$")

    ax_line.set_xlabel(r"$b\ (s/mm^2)$", fontsize=fs_ticks)
    ax_line.set_xlim(-50, 850)
    ax_line.set_ylim(-3, 0.5)
    ax_line.grid(alpha=0.25)
    ax_line.tick_params(axis="both", labelsize=fs_ticks)
    ax_line.spines.right.set_visible(False)
    ax_line.spines.top.set_visible(False)

    if i != 1:
        ax_line.set_yticklabels([])
    else:
        ax_line.legend(loc="lower left", fontsize=fs_legend)
        ax_line.set_ylabel(r"$ln(S/S_0)$", fontsize=fs_axtitle)

plot_fname = config.PLOT_DIR / "calibration_b_value_decay_plots_s0_dwi.png"
plt.savefig(plot_fname, dpi=600, format="png", bbox_inches="tight", pad_inches=0.1)

### B-VALUE MAPS

In [ ]:
def fill_frame(array, crop_region):
    x1, x2, y1, y2 = list(crop_region)
    return array[x1:x2, y1:y2]

dur = 0.035
plot_df = dwi_df[dwi_df["dwi_grad_duration"] == dur].copy()
names = plot_df["seq_name"].unique()
scan_spacing = plot_df["spacing"].values[0]

outplane_offset = 10
inplane_offsets = np.array([[0, 0], [0, -15], [-15, -12], [-22, 0], [-15, 12]])

n_row = len(list(plot_df["dwi_direction"].unique()))
n_img_col = 4
n_col = 5

fig, axes = plt.subplots(
    n_row,
    n_col,
    figsize=(15, 9),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1,1,1,1,2]},
)

fs_axtitle = 10
fs_plottitle = 16
fs_ticks = 10
fs_legend = 10

line_cmap = colorcet.cm.isolum
voxel_colors = [line_cmap(i) for i in [0, 0.25, 0.5, 0.75, 1.0]]
voxel_markers = ["o", "s", "^", "D", "v"]

vmin = 0.5
vmax = 1.5
norm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)

for i, dw_direction in enumerate(plot_df["dwi_direction"].unique()):

    direction_df = plot_df[plot_df["dwi_direction"]==dw_direction].copy()

    b_vals = []
    row_voxel_signals = {i: [] for i in range(inplane_offsets.shape[0])}
    for j, b in enumerate(sorted(direction_df["b_value"].unique())):

        ax = axes[i, j]

        subset_df = direction_df[direction_df["b_value"]==b].copy()
        b_clean = int(np.round(b, -2))
        cmap_avg = np.mean(subset_df["cmap_meas"].values, axis=0)
        cmap_bspline = cal.bspline_estimation(data=cmap_avg, spacing=scan_spacing, mask=preprocess_mask, spline_param=config.SPLINE_PARAM)

        img_centre = np.array(cmap_avg.shape) // 2
        plot_slice = cmap_bspline[:, :, img_centre[2] + outplane_offset]
        mask_slice = calib_mask[:, :, img_centre[2] + outplane_offset]
        plot_mask = np.full_like(plot_slice, np.nan, dtype=float)
        plot_mask[mask_slice == 0] = 1

        crop_lims = [2, -2, 2, -3]
        plot_slice = fill_frame(np.rot90(plot_slice, k=1), crop_lims)
        plot_mask = fill_frame(np.rot90(plot_mask, k=1), crop_lims)

        im = ax.imshow(
            plot_slice,
            norm=norm,
            cmap=colorcet.cm.coolwarm,
            interpolation=None,
        )
        ax.imshow(
            plot_mask,
            vmin=0,
            vmax=1,
            cmap="Greys_r",
            interpolation="nearest",
        )
        ax.set_axis_off()

        slice_centre = np.array(plot_slice.shape) // 2
        voxel_coords_2d = slice_centre + inplane_offsets

        # mark selected voxels on the image
        for k, (r, c) in enumerate(voxel_coords_2d):
            ax.plot(
                c,
                r,
                marker=voxel_markers[k],
                ms=5.5,
                color=voxel_colors[k],
                mec="k",
                mew=0.8,
                linestyle="None",
            )
        
        # collect voxel values for line plot
        b_vals.append(b_clean)
        for k, (r, c) in enumerate(voxel_coords_2d):
            row_voxel_signals[k].append(plot_slice[r, c])
        
        if i == 0:
            ax.set_title(rf"$b = {b_clean}$", fontsize=fs_plottitle, pad=12)

    # 5th column: voxel value vs b-value
    ax_line = axes[i, 4]
    b_vals = np.array(b_vals)
    sort_idx = np.argsort(b_vals)
    b_vals_sorted = b_vals[sort_idx]

    for i_voxel in range(inplane_offsets.shape[0]):

        vals = np.array(row_voxel_signals[i_voxel])[sort_idx]
        ax_line.plot(
            b_vals_sorted,
            vals,
            marker=voxel_markers[i_voxel],
            lw=1,
            ms=5.5,
            mec="k",
            mew=0.8,
            linestyle="--",
            color=voxel_colors[i_voxel],
            label=rf"$P_{{{i_voxel+1}}}$",
        )
    if i==2:
        ax_line.set_xlabel(r"$b\ (s/mm^2)$", fontsize=fs_axtitle)
    ax_line.set_xticks(b_vals_sorted)
    ax_line.set_ylabel(r"correction factor", fontsize=fs_axtitle)
    ax_line.grid(alpha=0.25)
    ax_line.set_xlim(b_vals_sorted.min() - 50, b_vals_sorted.max() + 50)
    ax_line.set_ylim(vmin, 1.3)

    if i == 0:
        ax_line.legend(loc="upper center", ncol=voxel_coords_2d.shape[0], fontsize=fs_legend)

    axes[i, 0].text(
        -0.05, 0.5,
        rf"${dw_direction}$",
        transform=axes[i, 0].transAxes,
        fontsize=fs_plottitle,
        fontweight="normal",
        ha="right",
        va="center",
    )

cbar = fig.colorbar(
    im,
    ax=axes[:, :n_img_col],
    pad=-0.05,
    location="bottom",
    orientation="horizontal",
    fraction=0.02,
    aspect=70,
)
cbar.set_label(r"correction factor", labelpad=8)
cbar.outline.set_linewidth(1)

fmt = "png"
plot_fname = config.PLOT_DIR / f"calibration_bvalue_maps_plus_voxel_line_plots.{fmt}"
fig.savefig( fname=plot_fname, dpi=600, format=fmt)